# Figure 1 — cohort flow

**Who was compared.** The target trial admits IVT-eligible patients at centres where
both strategies were actually used, so two [§3] design restrictions stand between the workbook and the
analysis cohort, and two [§11] denominators stand between the cohort and the estimate. A reader cannot
size the entry-conditioning selection effect [§2, §16] without seeing all four.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The flow, as Stage 5 recorded it

`cohort.build` writes the flow to the audit log as it restricts. Reading it back is how this figure
stays a view of the run rather than a second count of the same patients.

In [ ]:
flow = bundle.audit.entry("cohort", "cohort_flow").table
flow = pd.DataFrame(list(flow[1:]), columns=list(flow[0])).set_index("step")
flow

### The two [§11] denominators

The cohort is not the estimation population. A patient missing a confounder has no propensity score
and is not in the weighted contrast; a patient with no 90-day mRS is not in the primary estimate.

In [ ]:
def by_arm(mask):
    """`n (control / treated)` for a mask over the cohort frame."""
    a = bundle.cohort.loc[mask, C.TREATMENT]
    return int(mask.sum()), int((a == 0).sum()), int((a == 1).sum())


records = flow["records"].astype(int)
n_model, model_0, model_1 = by_arm(bundle.ps.in_model)
n_estimate, estimate_0, estimate_1 = by_arm(bundle.primary.in_estimate)

steps = [
    ("Records in the source workbook", records["as classified"],
     int(flow.loc["as classified", ARM[0]]), int(flow.loc["as classified", ARM[1]])),
    ("At a centre where both strategies were used", records["after restriction 1"],
     int(flow.loc["after restriction 1", ARM[0]]), int(flow.loc["after restriction 1", ARM[1]])),
    ("No absolute contraindication to IVT — the [§3] cohort", records["after restriction 2"],
     int(flow.loc["after restriction 2", ARM[0]]), int(flow.loc["after restriction 2", ARM[1]])),
    ("Confounders complete — in the propensity model", n_model, model_0, model_1),
    ("90-day mRS recorded — the primary estimate", n_estimate, estimate_0, estimate_1),
]

source = pd.DataFrame(steps, columns=["step", "records", ARM[0], ARM[1]])
source

## 3. The figure

Each box is a population and each annotation is the exclusion that separates it from the box above —
computed as the difference, so the arithmetic cannot drift from the counts.

In [ ]:
drops = [
    f"−{steps[0][1] - steps[1][1]} at centres that never gave IVT\n"
    "P(IVT = 1 | centre, X) = 0 structurally [§3]",
    f"−{steps[1][1] - steps[2][1]} absolute contraindication to IVT\n"
    "never candidates for bridging [§3]",
    f"−{steps[2][1] - steps[3][1]} incomplete confounder [§11]",
    f"−{steps[3][1] - steps[4][1]} no 90-day mRS [§11]",
]

fig, ax = plt.subplots(figsize=(7.0, 4.8))
M.flow(ax, [(label, f"n = {n}   ({n0} {ARM[0]} / {n1} {ARM[1]})") for label, n, n0, n1 in steps],
       drops)
fig.tight_layout()

## 4. Write it

The PNG is the exhibit; the CSV beside it carries the same counts and the statements the caption must
make. Both land in `out/manuscript/`.

In [ ]:
caption = (
    "Figure 1. Cohort flow. Both [§3] design restrictions — centres where bridging was never used, "
    "and patients with an absolute contraindication to IVT — and the two [§11] denominators that "
    "separate the cohort from the primary estimate. Counts are records (EVT alone / bridging).")

path = M.save_figure("Figure_1", fig, source, (
    caption,
    "Entry is conditional on these centres and on the eligibility restriction; the estimate is "
    "conditional on them too [§2, §10, §16].",
    "Every estimate carries its own denominator [§11]; the three nested populations are cohort, "
    "covariate-complete and outcome-present.",
    M.provenance_clause(bundle)))

M.save_legends()      # rebuilds out/manuscript/figure_legends.md
M.show(path)

## 5. The same flow as an editable PowerPoint diagram

A PNG is a picture of a diagram: a co-author cannot move a box in it. The cohort flow is the one
exhibit here that is pure geometry — no data-derived shape, only counts — so it is worth having twice.
This writes a slide-deck source whose boxes, arrows and annotations render as **native** PowerPoint
shapes, with every count taken from the same `steps` and `drops` above. Rendering it is a separate
command, in `spec.md` §8; nothing here shells out to another tool.

In [ ]:
import json

# Normalised to the slide's free slot: x, y are the box CENTRE, (0,0) is its top-left corner.
BOX_W, BOX_H, BOX_X = 0.50, 0.125, 0.27
FIRST, SPACING = 0.085, 0.205

nodes, edges, labels = [], [], []
for i, (label, n, n0, n1) in enumerate(steps):
    y = FIRST + i * SPACING
    final = i == len(steps) - 1
    nodes.append({"id": f"s{i}", "x": BOX_X, "y": y, "w": BOX_W, "h": BOX_H, "size": 12,
                  "text": f"{label}\nn = {n}   ({n0} {ARM[0]} / {n1} {ARM[1]})",
                  "fill": "teal_deep" if final else "panel_mist",
                  "color": "bg" if final else "text_primary"})
    if final:
        continue

    edges.append({"from": f"s{i}", "to": f"s{i + 1}", "arrow": "end"})
    labels.append({"text": drops[i], "x": 0.56, "y": y + SPACING / 2 - 0.075,
                   "color": "red", "size": 11})

diagram = {"caption": "Cohort flow — both [§3] design restrictions and the two [§11] denominators",
           "nodes": nodes, "edges": edges, "labels": labels}
len(nodes), len(edges), len(labels)

In [ ]:
deck = f"""---
title: Bridging versus EVT alone
subtitle: IVT before thrombectomy in late-window, CT-perfusion-selected stroke
year: 2026
---

## Cohort flow
- type: data
- notitle: true
- diagram: {json.dumps(diagram, ensure_ascii=False)}
"""

print(M.save_slide("Figure_1", deck))

## 6. Reading it

- **The first restriction is structural, not statistical.** At a centre where nobody received
  bridging, `P(IVT = 1 | centre, X) = 0`. No weighting recovers a contrast that was never available,
  which is why the centre is removed rather than adjusted for `[§3]`.
- **The second restriction defines the question.** Retaining contraindicated patients would make
  "no IVT" a marker of contraindications and of the prognosis that comes with them `[§3]`.
- **What the figure does not show** is who was eligible but never presented at these four centres.
  That selection is outside the data and is stated rather than estimated `[§2, §16]`.